# Import sempy and Fabric libraries

In [ ]:
import sempy
import sempy.fabric as fabric
import pandas

## Import Parameters

In [ ]:
%run 2. Parameters

## Setup Workspace and Dataset, then Run DAX Query

In [ ]:
tables_df = fabric.evaluate_dax(datasetId, dax_string=dax_query_tables, workspace=workspaceId)
tables_df.columns = [col.strip("[]").upper() for col in tables_df.columns]
spark_tables_df = spark.createDataFrame(tables_df)
display(spark_tables_df)

In [ ]:
# Write to lakehouse table
table_name = "TABLES"
# Convert to Spark DataFrame and write

spark_tables_df.write.mode("overwrite").format("delta").option("mergeSchema", "true").saveAsTable(table_name)

In [ ]:
columns_df = fabric.evaluate_dax(datasetId, dax_string=dax_query_columns, workspace=workspaceId)
columns_df.columns = [col.strip("[]").upper() for col in columns_df.columns]
spark_columns_df = spark.createDataFrame(columns_df)
display(spark_columns_df)

In [ ]:
# Set the datetime rebase mode
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")

# Write to lakehouse table
table_name = "COLUMNS"
# Convert to Spark DataFrame and write
spark_columns_df.write.mode("overwrite").format("delta").saveAsTable(table_name)

## List Columns

In [ ]:
from sempy import fabric

# Get the metadata including data types
list_columns_df = fabric.list_columns(dataset=datasetId, workspace=workspaceId)

# Add columns to your pandas DataFrame
list_columns_df['workspace_id'] = workspaceId
list_columns_df['dataset_id'] = datasetId
list_columns_df['database_name'] = semantic_model_name

# Automatically replace spaces with underscores in all column names
list_columns_df.columns = list_columns_df.columns.str.replace(' ', '_').str.upper()

# Drop columns with void/null datatypes
list_columns_df = list_columns_df.dropna(axis=1, how='all')

# Alternatively, explicitly drop the problematic column
# list_columns_df = list_columns_df.drop(columns=['sort_by_column'], errors='ignore')

# Define table name
table_name = "list_columns"

# Drop the table if it exists to clear any metadata issues
spark.sql(f"DROP TABLE IF EXISTS {table_name}")

# Convert to Spark DataFrame and write
spark_list_columns_df = spark.createDataFrame(list_columns_df)
spark_list_columns_df.write.mode("overwrite").format("delta").option("mergeSchema", "true").saveAsTable(table_name)

print(f"Successfully wrote {spark_list_columns_df.count()} rows to table '{table_name}'")

# Now query and display
df = spark.sql(f"SELECT * FROM {table_name} LIMIT 1000")
display(df)

In [ ]:
df = spark.sql("SELECT * FROM SMD_LH.list_columns LIMIT 1000")
display(df)